# 1. Magiv2 as it is

Magiv2 (ragavsachdeva/magiv2) reads a manga chapter: finds text, reads it (OCR), puts it in reading order,
throws away sound effects, and links every text to a character box.

(All the below code was taken from magiv2's documentation) 

In [1]:
%pip install -q "transformers==4.49.0" sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 78.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 79.9 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import numpy as np
import torch
from PIL import Image
from transformers import AutoModel

DATA = None
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    if "sequences.json" in files:
        DATA = root
print("data:", DATA)

seqs = json.load(open(DATA + "/sequences.json"))
print(len(seqs), "sequences")

data: /kaggle/input/datasets/arpitparashar06/manga-task/manga-task-upload/manga-task/dataset
95 sequences


In [3]:
model = AutoModel.from_pretrained("ragavsachdeva/magiv2", revision="fbc890fec52977142e8ee00bfe26e9458b65517c", trust_remote_code=True).cuda().eval()

def read_image(path):
    # grey then back to 3 channels, like the model card
    return np.array(Image.open(path).convert("L").convert("RGB"))

config.json: 0.00B [00:00, ?B/s]

configuration_magiv2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ragavsachdeva/magiv2:
- configuration_magiv2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modelling_magiv2.py: 0.00B [00:00, ?B/s]

processing_magiv2.py: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.06G [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": false,
  "transformers_version": "4.49.0"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

In [4]:
magi = {}
for s in seqs:
    pages = [read_image(DATA + "/" + p) for p in s["images"]]
    with torch.no_grad():
        results = model.do_chapter_wide_prediction(pages, {"images": [], "names": []}, use_tqdm=False, do_ocr=True)

    saved = []
    for r in results:
        saved.append({
            "text_boxes": [[float(v) for v in b] for b in r["texts"]],
            "char_boxes": [[float(v) for v in b] for b in r["characters"]],
            "links": [[int(a), int(b)] for a, b in r["text_character_associations"]],
            "tail_links": [[int(a), int(b)] for a, b in r["text_tail_associations"]],
            "clusters": [int(c) for c in r["character_cluster_labels"]],
            "keep": [bool(k) for k in r["is_essential_text"]],
            "ocr": [str(t) for t in r["ocr"]],
        })
    magi[s["sequence_id"]] = {"split": s["split"], "pages": saved}
    if len(magi) % 10 == 0:
        print(len(magi), "done")

json.dump(magi, open("/kaggle/working/magi_output.json", "w"))
print("saved magi_output.json")

10 done
20 done
30 done
40 done
50 done
60 done
70 done
80 done
90 done
saved magi_output.json


In [ ]:
# magi as it is, as predictions for the 80 dev sequences (cell 8 needs this file)
# speaker = magi's character group on that page, text with no link = NARRATION
f = open("/kaggle/working/magi_asis_dev.jsonl", "w")
for sid in magi:
    if magi[sid]["split"] == "test":
        continue
    pages = [[], [], []]
    for p in range(3):
        r = magi[sid]["pages"][p]
        link = {a: b for a, b in r["links"]}
        for i in range(len(r["ocr"])):
            if r["keep"][i] == True and r["ocr"][i].strip() != "":
                c = link.get(i, -1)
                if 0 <= c < len(r["clusters"]):
                    speaker = "group" + str(r["clusters"][c])
                else:
                    speaker = "NARRATION"
                pages[p].append({"speaker": speaker, "text": r["ocr"][i].strip()})
    f.write(json.dumps({"sequence_id": sid, "pages": pages}) + "\n")
f.close()

!python {DATA}/score.py --references {DATA}/development/labels.jsonl --predictions /kaggle/working/magi_asis_dev.jsonl --output /kaggle/working/magi_asis_dev_scores.json

In [5]:

r = magi[seqs[0]["sequence_id"]]["pages"][0]
link = {a: b for a, b in r["links"]}
for i in range(len(r["ocr"])):
    c = link.get(i, -1)
    group = r["clusters"][c] if 0 <= c < len(r["clusters"]) else "-"
    print(i, "keep" if r["keep"][i] else "drop", "group", group, "|", r["ocr"][i])

0 drop group 0 | Snap
1 keep group 0 | Huh? Does komaba seem different than usual?
2 keep group 0 | He's doing really well.
3 drop group 0 | http://www.youtube.com/watch?v=n=n=n-t-t-t-t-t-t-t-t-t-t-t-t-t-t-t-t-t-t-t-t-t-t-t
4 keep group 0 | It looks like he's been working hard for the fall tournament.
5 drop group 0 | The second
6 keep group 1 | No, that's not it. It's like he's...
7 drop group 1 | I
8 keep group 0 | out for blood.


## score on the Val set

speaker = magi's character group, text with no link = NARRATION

In [8]:

# the same 16 sequences used to validate the Qwen experiments
val16 = ["seq_07838fb26cd1be54", "seq_0ef2ec294597f572", "seq_4833e23eec54344b", "seq_58858a2227cf3761",
         "seq_5f407d0064bb4a43", "seq_84541a1f4b159018", "seq_96839890d69d7698", "seq_c4448514b6de1ab9",
         "seq_cd727990f22c777b", "seq_d8afceb0e8f9122d", "seq_d8bc3c6ac1bb6017", "seq_de542b63e6b19bc2",
         "seq_eaad741293da2ac1", "seq_ec05f45d71c43f2c", "seq_ee880b9612205860", "seq_fc4c2b84b6ec32ff"]

f = open("/kaggle/working/val16_labels.jsonl", "w")
for line in open(DATA + "/development/labels.jsonl"):
    if json.loads(line)["sequence_id"] in val16:
        f.write(line)
f.close()

f = open("/kaggle/working/magi_asis_val16.jsonl", "w")
for line in open("/kaggle/working/magi_asis_dev.jsonl"):
    if json.loads(line)["sequence_id"] in val16:
        f.write(line)
f.close()

!python {DATA}/score.py --references /kaggle/working/val16_labels.jsonl --predictions /kaggle/working/magi_asis_val16.jsonl --output /kaggle/working/magi_asis_val16_scores.json

{
  "text_order_score": 0.9407950002777625,
  "balanced_joint_f1": 0.27261202571103244,
  "speaker_accuracy_on_matched": 0.5022667415101362,
  "joint_f1": 0.4775124520698589,
  "matched_token_coverage": 0.9575425989366222
}
